# 11c - Soft report labels from an open-weights LLM

Top public notebooks train their image models on **probabilities** read from the
reports by an LLM, rather than yes/no labels: a hedged report ("suspected tear") should
become something like 0.8, not 1. This notebook builds those soft labels with
`Qwen/Qwen2.5-7B-Instruct` (Apache-2.0, multilingual) at a pinned revision.

For each report and finding, the model is asked whether the report describes the
finding; the label is P(yes) / (P(yes) + P(no)) from the next-token logits. Nothing is
generated, and the report prefix is encoded once and reused for the 12 questions.

* `STAGE = "pilot"`: only the 58 expert-labelled studies. Agreement AUC against the
  expert labels next to the regex extractor (notebook 11) and a regex+LLM mean, plus
  measured throughput and a projected full-run time.
* `STAGE = "full"`: all 4,407 studies -> `soft_labels_v1.csv`.

The label source written to `soft_labels_v1.csv` (LLM alone, or the regex+LLM mean) is
chosen by agreement on the 58 expert-labelled studies; that choice uses the same
studies that later evaluate the image model, which is stated in the report.

**Settings:** GPU T4 x2 (fp16), or a single 16 GB GPU (then the model loads in 4-bit
NF4, recorded in the metadata); internet ON (model download, about 15 GB).
**Inputs:** competition `train.csv`; notebook 11 `weak_labels_v1.csv`.
**Outputs:** `soft_labels_v1.csv`, `soft_label_agreement_v1.csv`, `soft_labels_meta_v1.json`.
No report text or identifiers are printed or saved.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# The competition mount path has differed between Kaggle runtimes; take the first that has train.csv.
_SLUG = "rsna-knee-abnormality-detection"
_roots = [Path("/kaggle/input/competitions") / _SLUG, Path("/kaggle/input") / _SLUG,
          *sorted(Path("/kaggle/input").glob("*/" + _SLUG)), *sorted(Path("/kaggle/input").glob("*"))]
ROOT = next((r for r in _roots if (r / "train.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("competition train.csv not found; /kaggle/input has: "
                            + str(sorted(str(p) for p in Path("/kaggle/input").glob("*/*"))[:50]))
print("competition data:", ROOT)
OUT = Path("/kaggle/working")
TARGET_COLUMNS = [
    "ACL", "MCL",
    "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]
SEED = 42
from pathlib import Path


def find_input(name):
    """Locate one attached upstream file by name.

    Only notebook outputs and datasets are searched: walking the competition
    mount would visit millions of DICOM files.
    """
    hits = []
    for base in (Path("/kaggle/input/notebooks"), Path("/kaggle/input/datasets"),
                 Path("/kaggle/input")):
        if base.is_dir():
            for child in base.iterdir():
                if (child.is_dir()
                        and child.name not in ("competitions", "notebooks", "datasets")
                        and not (child / "train_series").exists()
                        and not (child / "test_series").exists()):
                    hits += list(child.rglob(name))
    hits = sorted(set(hits))
    if len(hits) != 1:
        raise FileNotFoundError(
            f"expected exactly one attached '{name}', found {len(hits)}: {hits[:5]}. "
            "Attach the upstream notebook output via Add Input."
        )
    return hits[0]

import json
import time

import transformers
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

STAGE = "full"  # "pilot" (58 expert-labelled studies) or "full" (all 4,407)
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MODEL_REVISION = "a09a35458c702b33eeacc393d103063234e8bc28"
MAX_REPORT_CHARS = 6000  # longer reports are cut (counted below)
QUESTIONS = {
    "ACL": "anterior cruciate ligament (ACL) tear (high-grade partial or complete)",
    "MCL": "medial collateral ligament (MCL) tear (high-grade partial or complete)",
    "Medial Meniscus": "medial meniscus tear",
    "Lateral Meniscus": "lateral meniscus tear",
    "Medial OA": "medial tibiofemoral compartment osteoarthritis (substantial cartilage loss)",
    "Lateral OA": "lateral tibiofemoral compartment osteoarthritis (substantial cartilage loss)",
    "PF OA": "patellofemoral osteoarthritis (substantial cartilage loss)",
    "Effusion": "joint effusion",
    "Synovitis": "synovitis",
    "Baker's": "Baker's (popliteal) cyst",
    "Contusion": "bone contusion / bone bruise (traumatic bone marrow edema)",
    "Fracture": "fracture"
}
assert list(QUESTIONS) == TARGET_COLUMNS
print("transformers", transformers.__version__, "| stage", STAGE)

In [ ]:
# Soft report labels: P(yes) for "is <finding> present?", read from the next-token
# logits (no generation). The report prefix is encoded once and its KV cache reused
# for the 12 finding questions.
import torch

SYSTEM = ("You are a musculoskeletal radiologist. You read knee MRI reports, "
          "which may be written in any language.")


def build_prompt(tokenizer, report, finding):
    user = (f"Knee MRI report:\n{report}\n\n"
            f"Question: does this report describe {finding}? "
            "Answer yes if it is stated or suspected; answer no if it is described as "
            "absent or normal, or not mentioned. Answer with one word: yes or no.")
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True)


def answer_token_ids(tokenizer):
    """Single-token spellings of yes/no (case and leading-space variants)."""
    ids = {}
    for key, words in (("yes", ("yes", "Yes", " yes", " Yes")), ("no", ("no", "No", " no", " No"))):
        toks = set()
        for w in words:
            enc = tokenizer.encode(w, add_special_tokens=False)
            if len(enc) == 1:
                toks.add(enc[0])
        assert toks, f"no single-token spelling for {key}"
        ids[key] = sorted(toks)
    return ids


def p_yes(last_logits, ids):
    """last_logits: (n, vocab) -> P(yes) / (P(yes) + P(no)) per row."""
    lp = torch.log_softmax(last_logits.float(), dim=-1)
    yes = torch.logsumexp(lp[:, ids["yes"]], dim=-1)
    no = torch.logsumexp(lp[:, ids["no"]], dim=-1)
    return torch.sigmoid(yes - no)


def common_prefix_len(seqs):
    n = min(len(s) for s in seqs)
    for i in range(n):
        if any(s[i] != seqs[0][i] for s in seqs[1:]):
            return i
    return n


@torch.inference_mode()
def score_report_naive(model, tokenizer, report, findings, ids):
    """Reference path: every (report, finding) prompt encoded in full, left-padded."""
    texts = [build_prompt(tokenizer, report, f) for f in findings]
    tokenizer.padding_side = "left"
    batch = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
    logits = model(**batch).logits[:, -1, :]
    return p_yes(logits, ids).cpu().numpy()


@torch.inference_mode()
def score_report_cached(model, tokenizer, report, findings, ids):
    """Fast path: shared prefix once, then the 12 right-padded suffixes on its KV cache."""
    seqs = [tokenizer(build_prompt(tokenizer, report, f), add_special_tokens=False)["input_ids"]
            for f in findings]
    k = common_prefix_len(seqs) - 1  # keep >= 1 suffix token per row
    prefix = torch.tensor([seqs[0][:k]], device=model.device)
    cache = model(input_ids=prefix, use_cache=True).past_key_values
    cache.batch_repeat_interleave(len(seqs))
    suffix = [s[k:] for s in seqs]
    width = max(len(s) for s in suffix)
    pad = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    ids_t = torch.tensor([s + [pad] * (width - len(s)) for s in suffix], device=model.device)
    mask = torch.tensor([[1] * (k + len(s)) + [0] * (width - len(s)) for s in suffix],
                        device=model.device)
    out = model(input_ids=ids_t, attention_mask=mask, past_key_values=cache,
                cache_position=torch.arange(k, k + width, device=model.device))
    last = torch.tensor([len(s) - 1 for s in suffix], device=model.device)
    logits = out.logits[torch.arange(len(suffix), device=model.device), last]
    return p_yes(logits, ids).cpu().numpy()

In [ ]:
# Data-free checks of the scoring maths.
_ids = {"yes": [1, 3], "no": [2]}
_l = torch.full((2, 5), -20.0)
_l[0, 1] = 5.0; _l[0, 2] = 0.0   # mostly yes
_l[1, 1] = 0.0; _l[1, 2] = 0.0   # even split between one yes spelling and no
_p = p_yes(_l, _ids).numpy()
assert _p[0] > 0.99 and abs(_p[1] - 0.5) < 1e-3, _p
assert common_prefix_len([[1, 2, 3, 4], [1, 2, 5]]) == 2
assert common_prefix_len([[7, 8], [7, 8]]) == 2
print("soft-scorer self-test passed")

## 1. Inputs

In [ ]:
train = pd.read_csv(ROOT / "train.csv", dtype={"StudyInstanceUID": str})
weak = pd.read_csv(find_input("weak_labels_v1.csv"), dtype={"StudyInstanceUID": str})
data = train[["StudyInstanceUID", "Report"]].merge(weak, on="StudyInstanceUID", validate="one_to_one")
data["Report"] = data["Report"].map(lambda r: " ".join(str(r).split()))
print("reports longer than the cut:", int(data["Report"].str.len().gt(MAX_REPORT_CHARS).sum()))
data["Report"] = data["Report"].str.slice(0, MAX_REPORT_CHARS)
todo = data[data["is_gold"].eq(1)] if STAGE == "pilot" else data
print("studies to score:", len(todo))

## 2. Model

In [ ]:
assert torch.cuda.is_available(), "enable the GPU T4 x2 accelerator"
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
gpu_gb = sum(torch.cuda.get_device_properties(i).total_memory
             for i in range(torch.cuda.device_count())) / 1e9
QUANTISATION = "fp16" if gpu_gb >= 24 else "nf4-4bit"  # 7B fp16 (~15 GB) does not fit one 16 GB GPU
print(f"total GPU memory {gpu_gb:.0f} GB -> {QUANTISATION}")
load_kw = {"torch_dtype": torch.float16, "device_map": "auto"}
if QUANTISATION != "fp16":
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=True)
    from transformers import BitsAndBytesConfig
    load_kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REVISION, **load_kw).eval()
ANSWER_IDS = answer_token_ids(tokenizer)
print("answer token ids:", ANSWER_IDS)

# The cached fast path must agree with the plain path before it is trusted.
_r = data.loc[data["is_gold"].eq(1), "Report"].iloc[0]
_findings = [QUESTIONS[t] for t in TARGET_COLUMNS]
_a = score_report_naive(model, tokenizer, _r, _findings, ANSWER_IDS)
try:
    _b = score_report_cached(model, tokenizer, _r, _findings, ANSWER_IDS)
    print("cached vs plain, max |diff|:", float(np.abs(_a - _b).max()))
    assert np.abs(_a - _b).max() < 0.02
    score_report = score_report_cached
except Exception as exc:  # cache API differs across transformers versions
    print(f"cached path unavailable ({type(exc).__name__}: {exc}); using the plain path")
    score_report = score_report_naive

## 3. Score

In [ ]:
FINDINGS = [QUESTIONS[t] for t in TARGET_COLUMNS]
part_path = OUT / f"soft_llm_{STAGE}_partial.csv"
done = pd.read_csv(part_path, dtype={"StudyInstanceUID": str}) if part_path.exists() else None
rows = done.to_dict("records") if done is not None else []
seen = {r["StudyInstanceUID"] for r in rows}
t0, n_new = time.time(), 0
for i, row in enumerate(tqdm(todo.itertuples(index=False), total=len(todo), desc="reports",
                             mininterval=30), 1):
    if row.StudyInstanceUID in seen:
        continue
    p = score_report(model, tokenizer, row.Report, FINDINGS, ANSWER_IDS)
    rows.append({"StudyInstanceUID": row.StudyInstanceUID, **dict(zip(TARGET_COLUMNS, p.tolist()))})
    n_new += 1
    if n_new % 200 == 0:
        pd.DataFrame(rows).to_csv(part_path, index=False)
pd.DataFrame(rows).to_csv(part_path, index=False)
secs = time.time() - t0
print(f"scored {n_new} reports in {secs:.0f}s ({n_new / max(secs, 1e-9):.2f} reports/s)")
if STAGE == "pilot":
    print(f"projected full run (4,407 reports): {4407 * secs / max(n_new, 1) / 3600:.2f} h")
llm = pd.DataFrame(rows).set_index("StudyInstanceUID")[TARGET_COLUMNS]
assert llm.notna().all().all() and ((llm >= 0) & (llm <= 1)).all().all()

## 4. Agreement with the expert labels

In [ ]:
from sklearn.metrics import roc_auc_score


def safe_auc(y, p):
    """AUC, or NaN when only one class is present (the metric is undefined)."""
    y = np.asarray(y)
    return float(roc_auc_score(y, p)) if np.unique(y).size == 2 else np.nan


def auc_table(gold, scores, n_boot=2000, seed=0):
    """Per-target AUC with a percentile bootstrap interval over studies."""
    rng = np.random.default_rng(seed)
    rows = []
    for t in TARGET_COLUMNS:
        y, p = gold[t].to_numpy(), scores[t].to_numpy()
        boots = []
        for _ in range(n_boot):
            i = rng.integers(0, len(y), len(y))
            a = safe_auc(y[i], p[i])
            if np.isfinite(a):
                boots.append(a)
        rows.append({"target": t, "auc": safe_auc(y, p),
                     "ci_low": np.percentile(boots, 2.5) if boots else np.nan,
                     "ci_high": np.percentile(boots, 97.5) if boots else np.nan,
                     "n_pos": int(y.sum()), "n_neg": int((1 - y).sum())})
    return pd.DataFrame(rows)


def macro_auc(table):
    """Challenge metric: mean of per-target AUCs; undefined targets excluded and counted."""
    return float(table["auc"].mean(skipna=True)), int(table["auc"].notna().sum())

g = data[data["is_gold"].eq(1)].set_index("StudyInstanceUID")
gold_y = g[[t + "__gold" for t in TARGET_COLUMNS]].set_axis(TARGET_COLUMNS, axis=1).astype(int)
regex = g[[t + "__weak" for t in TARGET_COLUMNS]].set_axis(TARGET_COLUMNS, axis=1)
llm_g = llm.loc[g.index]
tables = {"llm": auc_table(gold_y, llm_g), "regex": auc_table(gold_y, regex),
          "blend": auc_table(gold_y, (llm_g + regex) / 2)}
agreement = pd.DataFrame({"target": TARGET_COLUMNS,
                          **{f"{k}_auc": v["auc"] for k, v in tables.items()},
                          **{f"{k}_ci_low": v["ci_low"] for k, v in tables.items()},
                          **{f"{k}_ci_high": v["ci_high"] for k, v in tables.items()},
                          "n_pos": tables["llm"]["n_pos"], "n_neg": tables["llm"]["n_neg"]})
display(agreement[["target", "llm_auc", "regex_auc", "blend_auc", "n_pos", "n_neg"]].round(3))
macros = {k: macro_auc(v)[0] for k, v in tables.items()}
print("macro AUC:", {k: round(v, 4) for k, v in macros.items()})
SOURCE = "blend" if macros["blend"] > max(macros["llm"], macros["regex"]) else "llm"
print("label source chosen for training:", SOURCE)
print("NOTE: 58 studies -> differences under ~0.05 are within noise.")

## 5. Save

In [ ]:
agreement.to_csv(OUT / "soft_label_agreement_v1.csv", index=False)
meta = {"stage": STAGE, "model": MODEL_ID, "revision": MODEL_REVISION, "quantisation": QUANTISATION,
        "source": SOURCE,
        "macro_auc": macros, "n_reports": int(len(llm)), "seconds": secs,
        "max_report_chars": MAX_REPORT_CHARS, "transformers": transformers.__version__}
(OUT / "soft_labels_meta_v1.json").write_text(json.dumps(meta, indent=2))
if STAGE == "full":
    reg_all = data.set_index("StudyInstanceUID")[[t + "__weak" for t in TARGET_COLUMNS]]
    reg_all = reg_all.set_axis(TARGET_COLUMNS, axis=1).loc[llm.index]
    soft = (llm + reg_all) / 2 if SOURCE == "blend" else llm
    out = soft.reset_index()
    for t in TARGET_COLUMNS:
        out[t + "__llm"] = llm[t].to_numpy()
    assert len(out) == 4407 and "Report" not in out.columns
    out.to_csv(OUT / "soft_labels_v1.csv", index=False)
    print("saved soft_labels_v1.csv", out.shape, "| source:", SOURCE)
print(json.dumps(meta, indent=2))